# 🔍 Fraud Detection — Complete EDA Notebook

**Purpose:** This notebook provides the Exploratory Data Analysis (EDA) that **motivated and validates** every preprocessing and feature engineering decision made in `src/preprocessing/` and `src/feature_engineering/`.

---

## Sections Covered

| Section | Justifies |
|---|---|
| 1. Dataset Overview | load_raw_data, merge strategy |
| 2. Target Distribution | class imbalance, fraud rate |
| 3. Temporal Analysis | sort_by_time, time_split, cyclic encoding |
| 4. Missing Values Deep Dive | missing_values.py strategies per model |
| 5. V-Column Analysis | v_column_reduction.py — NaN groups, correlation, PCA |
| 6. M-Column Analysis | encode_m_columns, build_m_features |
| 7. C-Column Analysis | c_features.py — ratios, C3 flag |
| 8. D-Column Analysis | d_features.py — normalisation |
| 9. Transaction Amount Analysis | amount_features.py — log, decimal, z-score |
| 10. Email & Device Analysis | email_features.py, device_features.py |
| 11. UID Construction Validation | uid_features.py — card+addr+D1 trick |
| 12. Frequency Encoding Justification | frequency_encoding.py |
| 13. Velocity / Behavioral Features | velocity_features.py |
| 14. Cohort Prior Analysis | build_cohort_priors.py |
| 15. Feature Importance Preview | correlation with isFraud |


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import json
import re

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'text.color':       '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titlecolor':  '#58a6ff',
})
FRAUD_COLOR = '#ff6b6b'
LEGIT_COLOR = '#4ecdc4'
ACCENT     = '#58a6ff'

print('✅ Imports done')

In [ ]:
# ── Load Raw Data ─────────────────────────────────────────────────────────────
RAW = Path('../data/raw')

train_txn = pd.read_csv(RAW / 'train_transaction.csv')
train_id  = pd.read_csv(RAW / 'train_identity.csv')

# Left-join: keep all transactions, attach identity if available
train = train_txn.merge(train_id, on='TransactionID', how='left')

print(f'Transaction table : {train_txn.shape}')
print(f'Identity table    : {train_id.shape}')
print(f'Merged train      : {train.shape}')
print(f'Rows WITH identity: {train_id["TransactionID"].nunique():,} / {len(train):,}')

---
## Section 1 — Dataset Overview

In [ ]:
# Column-type summary
col_types = {
    'V-columns (V1–V339)':   [c for c in train.columns if c.startswith('V') and c[1:].isdigit()],
    'C-columns (C1–C14)':    [c for c in train.columns if c.startswith('C') and c[1:].isdigit()],
    'D-columns (D1–D15)':    [c for c in train.columns if c.startswith('D') and c[1:].isdigit()],
    'M-columns (M1–M9)':     [c for c in train.columns if c.startswith('M') and c[1:].isdigit()],
    'id_columns (id_01–id_38)': [c for c in train.columns if c.startswith('id_')],
    'Card columns':          [c for c in train.columns if c.startswith('card')],
    'Other features':        ['TransactionAmt','ProductCD','addr1','addr2',
                              'P_emaildomain','R_emaildomain','DeviceType','DeviceInfo'],
}

fig, ax = plt.subplots(figsize=(10, 4))
sizes  = [len(v) for v in col_types.values()]
labels = list(col_types.keys())
colors = ['#58a6ff','#bc8cff','#3fb950','#ff6b6b','#f0883e','#4ecdc4','#e3b341']
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.0f%%', startangle=140,
    textprops={'color': '#c9d1d9', 'fontsize': 9},
)
for at in autotexts:
    at.set_color('white')
    at.set_fontsize(8)
ax.set_title('Column Composition of the Merged Dataset', pad=15)
plt.tight_layout()
plt.show()

for k, v in col_types.items():
    print(f'  {k:35s}: {len(v):3d} columns')

---
## Section 2 — Target Distribution & Class Imbalance

In [ ]:
fraud_rate = train['isFraud'].mean()
fraud_cnt  = train['isFraud'].sum()
legit_cnt  = len(train) - fraud_cnt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(['Legitimate', 'Fraud'], [legit_cnt, fraud_cnt],
                   color=[LEGIT_COLOR, FRAUD_COLOR], width=0.5, edgecolor='white', linewidth=0.5)
axes[0].set_title('Transaction Count by Class')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, [legit_cnt, fraud_cnt]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{val:,}', ha='center', color='#c9d1d9', fontsize=10)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# Pie chart
axes[1].pie([legit_cnt, fraud_cnt], labels=['Legitimate\n96.5%', 'Fraud\n3.5%'],
            colors=[LEGIT_COLOR, FRAUD_COLOR], startangle=90,
            textprops={'color': '#c9d1d9'})
axes[1].set_title(f'Class Imbalance — Fraud Rate: {fraud_rate:.2%}')

plt.suptitle('⚠️  Severely Imbalanced Dataset — Motivates Careful Evaluation (AUC-ROC, not Accuracy)',
             color='#f0883e', fontsize=10)
plt.tight_layout()
plt.show()

print(f'Total transactions : {len(train):,}')
print(f'Fraud transactions : {fraud_cnt:,}  ({fraud_rate:.3%})')
print(f'Imbalance ratio    : 1 : {legit_cnt/fraud_cnt:.0f}')

---
## Section 3 — Temporal Analysis
> **Justifies:** `sort_by_time()`, `time_split()`, `expanding_window_folds()`, `build_time_features()`

In [ ]:
# ── Sort by TransactionDT (THE FIRST THING WE DO) ────────────────────────────
train = train.sort_values('TransactionDT').reset_index(drop=True)
train['day']     = train['TransactionDT'] // 86400
train['hour']    = (train['TransactionDT'] // 3600) % 24
train['weekday'] = train['day'] % 7

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Transaction volume over time
daily = train.groupby('day').agg(total=('isFraud','count'), fraud=('isFraud','sum')).reset_index()
axes[0].plot(daily['day'], daily['total'], color=LEGIT_COLOR, lw=1.5, label='All Txns')
ax0r = axes[0].twinx()
ax0r.plot(daily['day'], daily['fraud'] / daily['total'], color=FRAUD_COLOR, lw=1.5, label='Fraud Rate')
axes[0].set_title('Daily Transaction Volume & Fraud Rate')
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Total Transactions', color=LEGIT_COLOR)
ax0r.set_ylabel('Fraud Rate', color=FRAUD_COLOR)

# Split point
split_day = int(train['day'].quantile(0.75))
axes[0].axvline(split_day, color='#f0883e', lw=2, ls='--', label=f'Train/Val split (day {split_day})')
axes[0].legend(loc='upper left', fontsize=8)

# Fraud rate by hour
hourly = train.groupby('hour')['isFraud'].mean()
axes[1].bar(hourly.index, hourly.values * 100, color=ACCENT)
axes[1].axhspan(0, 5, color=FRAUD_COLOR, alpha=0.08)
axes[1].axhspan(22, 100, color=FRAUD_COLOR, alpha=0.08)
axes[1].set_title('Fraud Rate by Hour of Day')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].text(23, hourly.max()*100*0.95, '← Late night\n   higher fraud', color=FRAUD_COLOR, fontsize=8)

# Fraud rate by weekday
day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
weekly = train.groupby('weekday')['isFraud'].mean()
colors_wk = [FRAUD_COLOR if i >= 5 else ACCENT for i in range(7)]
axes[2].bar([day_names[i] for i in weekly.index], weekly.values * 100, color=colors_wk)
axes[2].set_title('Fraud Rate by Day of Week')
axes[2].set_xlabel('Day')
axes[2].set_ylabel('Fraud Rate (%)')

plt.suptitle('Temporal Patterns → Justifies Time-Based Split + Cyclic Encoding', color='#58a6ff')
plt.tight_layout()
plt.show()

In [ ]:
# ── WHY CYCLIC ENCODING? ─────────────────────────────────────────────────────
hours = np.arange(24)
h_sin = np.sin(2 * np.pi * hours / 24)
h_cos = np.cos(2 * np.pi * hours / 24)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw integer: hour 23 and 0 are far apart
sc = axes[0].scatter(hours, hours, c=hours, cmap='plasma', s=100, zorder=3)
axes[0].set_title('❌ Plain Integer Encoding\nHour 23 and 0 are 23 units apart!')
axes[0].set_xlabel('Hour value')
axes[0].set_ylabel('Hour value')
axes[0].annotate('Hour 23', (23, 23), xytext=(18, 21), color=FRAUD_COLOR,
                 arrowprops=dict(arrowstyle='->', color=FRAUD_COLOR))
axes[0].annotate('Hour 0', (0, 0), xytext=(5, 2), color=LEGIT_COLOR,
                 arrowprops=dict(arrowstyle='->', color=LEGIT_COLOR))

# Cyclic: hour 23 and 0 are adjacent on the circle
axes[1].scatter(h_cos, h_sin, c=hours, cmap='plasma', s=100, zorder=3)
for i in [0, 6, 12, 18, 23]:
    axes[1].annotate(f'Hr {i}', (h_cos[i], h_sin[i]),
                     textcoords='offset points', xytext=(5, 5),
                     color='white', fontsize=8)
axes[1].set_title('✅ Cyclic sin/cos Encoding\nHour 23 and 0 are adjacent on the circle!')
axes[1].set_xlabel('cos(2π × hour / 24)')
axes[1].set_ylabel('sin(2π × hour / 24)')
axes[1].set_aspect('equal')

plt.suptitle('Cyclic Encoding Preserves Temporal Adjacency', color='#58a6ff')
plt.tight_layout()
plt.show()
print('Hour 23 → 0 distance (plain):  23')
print(f'Hour 23 → 0 distance (cyclic): {np.sqrt((h_sin[23]-h_sin[0])**2 + (h_cos[23]-h_cos[0])**2):.3f}')

---
## Section 4 — Missing Values Deep Dive
> **Justifies:** `encode_m_columns()`, `fill_nan_for_trees(-999)`, `fit_median_imputer()`, `fill_nan_for_catboost()`

In [ ]:
# Missing rate per column family
v_cols  = [c for c in train.columns if c.startswith('V') and c[1:].isdigit()]
c_cols  = [c for c in train.columns if c.startswith('C') and c[1:].isdigit()]
d_cols  = [c for c in train.columns if c.startswith('D') and c[1:].isdigit()]
m_cols  = [c for c in train.columns if c.startswith('M') and c[1:].isdigit()]
id_cols = [c for c in train.columns if c.startswith('id_')]

def miss_rate(cols):
    return train[cols].isna().mean().values if cols else np.array([])

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

groups = [
    ('V columns (V1–V339)', v_cols, '#58a6ff'),
    ('C columns (C1–C14)',  c_cols, '#3fb950'),
    ('D columns (D1–D15)',  d_cols, '#f0883e'),
    ('M columns (M1–M9)',   m_cols, '#bc8cff'),
    ('id_ columns',         id_cols,'#4ecdc4'),
]

for ax, (title, cols, color) in zip(axes, groups):
    if not cols:
        ax.axis('off')
        continue
    rates = miss_rate(cols) * 100
    ax.hist(rates, bins=20, color=color, edgecolor='black', lw=0.3)
    ax.axvline(rates.mean(), color='white', lw=1.5, ls='--', label=f'Mean={rates.mean():.0f}%')
    ax.set_title(f'{title}\n(n={len(cols)} cols)')
    ax.set_xlabel('Missing Rate (%)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

# Overall top-20 worst columns
overall_miss = train.isna().mean().sort_values(ascending=False).head(20)
axes[5].barh(overall_miss.index, overall_miss.values * 100, color=FRAUD_COLOR)
axes[5].set_title('Top 20 Columns by Missing Rate')
axes[5].set_xlabel('Missing Rate (%)')
axes[5].invert_yaxis()

plt.suptitle('Missing Value Analysis → Motivates Different NaN Strategies per Model', color='#58a6ff')
plt.tight_layout()
plt.show()

In [ ]:
# ── WHY -999 for tree models? ─────────────────────────────────────────────────
# Show that fraud rate differs between present and missing values
key_cols = ['V258', 'C1', 'D1', 'id_02']
key_cols = [c for c in key_cols if c in train.columns]

fig, axes = plt.subplots(1, len(key_cols), figsize=(14, 4))
if len(key_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, key_cols):
    present_fraud = train.loc[train[col].notna(), 'isFraud'].mean() * 100
    missing_fraud = train.loc[train[col].isna(),  'isFraud'].mean() * 100
    bars = ax.bar(['Present', 'Missing'], [present_fraud, missing_fraud],
                  color=[LEGIT_COLOR, FRAUD_COLOR], edgecolor='white', lw=0.5)
    for bar, val in zip(bars, [present_fraud, missing_fraud]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}%', ha='center', color='white', fontsize=9)
    ax.set_title(f'{col}')
    ax.set_ylabel('Fraud Rate (%)')
    missing_pct = train[col].isna().mean() * 100
    ax.set_xlabel(f'({missing_pct:.0f}% missing)')

plt.suptitle('\"Missingness\" Itself Is a Fraud Signal → Trees Need -999 to Learn It', color='#58a6ff')
plt.tight_layout()
plt.show()

In [ ]:
# ── M column encoding: T/F strings → 1/0/-1 ──────────────────────────────────
m_present = [c for c in m_cols if c in train.columns]
print('M column value distributions BEFORE encoding:')
for m in m_present[:4]:
    print(f'  {m}: {dict(train[m].value_counts(dropna=False).head(5))}')

# Show that fraud rate differs between T and F
fig, axes = plt.subplots(1, min(4, len(m_present)), figsize=(14, 4))
if len(m_present) == 1:
    axes = [axes]

for ax, m in zip(axes, m_present[:4]):
    fraud_by_m = train.groupby(m, dropna=False)['isFraud'].mean() * 100
    colors_m = {v: FRAUD_COLOR if v in ['F', np.nan] else LEGIT_COLOR for v in fraud_by_m.index}
    bars = ax.bar(fraud_by_m.index.astype(str), fraud_by_m.values,
                  color=[FRAUD_COLOR, LEGIT_COLOR, '#888888'][:len(fraud_by_m)])
    ax.set_title(f'Fraud Rate by {m} value')
    ax.set_xlabel('Value (T=match, F=mismatch, NaN=unknown)')
    ax.set_ylabel('Fraud Rate (%)')

plt.suptitle('M Columns: T=Match, F=Mismatch, NaN=Unknown → Encode as 1/0/-1', color='#58a6ff')
plt.tight_layout()
plt.show()

---
## Section 5 — V-Column Analysis
> **Justifies:** `group_v_cols_by_nan()`, `reduce_v_by_correlation()`, `fit_v_pca()`, `add_uid_v_aggregations()`

In [ ]:
# ── V-column NaN grouping ─────────────────────────────────────────────────────
nan_counts = train[v_cols].isna().sum()
unique_nan_counts = nan_counts.value_counts().sort_index()

# Group V columns by NaN count (sensor block proxy)
from collections import defaultdict
groups = defaultdict(list)
for col in v_cols:
    groups[int(nan_counts[col])].append(col)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NaN distribution across V columns
axes[0].hist(nan_counts.values / len(train) * 100, bins=30, color=ACCENT, edgecolor='black', lw=0.3)
axes[0].set_title(f'V-Column Missing Rate Distribution\n({len(v_cols)} columns)')
axes[0].set_xlabel('Missing Rate (%)')
axes[0].set_ylabel('Number of V Columns')

# Group sizes
group_sizes = sorted([(k, len(v)) for k, v in groups.items()])
keys = [f'{k/len(train)*100:.0f}%\nNaN' for k, _ in group_sizes]
vals = [v for _, v in group_sizes]
axes[1].bar(range(len(keys)), vals, color='#bc8cff', edgecolor='black', lw=0.3)
axes[1].set_xticks(range(len(keys)))
axes[1].set_xticklabels(keys, fontsize=7)
axes[1].set_title(f'V Columns Grouped by NaN Pattern\n({len(groups)} distinct sensor blocks)')
axes[1].set_xlabel('NaN Rate Group')
axes[1].set_ylabel('Columns in Group')

plt.suptitle('V Columns Cluster into NaN Blocks → Same Sensor = Same Group → Reduce Within Groups', color='#58a6ff')
plt.tight_layout()
plt.show()

print(f'Total V columns    : {len(v_cols)}')
print(f'Distinct NaN groups: {len(groups)}')
print('Group sizes:', sorted([len(v) for v in groups.values()], reverse=True)[:10], '...')

In [ ]:
# ── Correlation within a V NaN group ─────────────────────────────────────────
# Pick the largest group
largest_group_key = max(groups, key=lambda k: len(groups[k]))
largest_group = groups[largest_group_key][:15]  # cap for display

sub = train[largest_group].fillna(train[largest_group].median())
corr_matrix = sub.corr().abs()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full correlation heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, ax=axes[0], cmap='YlOrRd', vmin=0, vmax=1,
            linewidths=0.3, mask=mask, annot=len(largest_group) <= 8, fmt='.1f')
axes[0].set_title(f'Pairwise Correlation — NaN Group {largest_group_key/len(train)*100:.0f}% missing\n({len(groups[largest_group_key])} cols, showing first 15)')

# Distribution of pairwise correlations
corr_vals = corr_matrix.values[np.tril(np.ones_like(corr_matrix, dtype=bool), k=-1)]
axes[1].hist(corr_vals, bins=20, color=ACCENT, edgecolor='black')
axes[1].axvline(0.9, color=FRAUD_COLOR, lw=2, ls='--', label='Threshold=0.9')
high_corr_pct = (corr_vals > 0.9).mean() * 100
axes[1].set_title(f'Distribution of Pairwise Correlations\n{high_corr_pct:.0f}% pairs above threshold')
axes[1].set_xlabel('|Pearson Correlation|')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('High Correlation Within V Groups → Greedy Correlation Reduction Removes Redundancy', color='#58a6ff')
plt.tight_layout()
plt.show()

In [ ]:
# ── PCA variance explained per NaN group ─────────────────────────────────────
pca_results = []
for nan_count, group in list(groups.items())[:8]:
    if len(group) < 3:
        continue
    X = train[group].fillna(0).values
    n_comp = min(5, len(group))
    pca = PCA(n_components=n_comp, random_state=42)
    pca.fit(X)
    pca_results.append({
        'group': f'{nan_count/len(train)*100:.0f}%NaN',
        'n_cols': len(group),
        'var_3': pca.explained_variance_ratio_[:3].sum(),
    })

df_pca = pd.DataFrame(pca_results)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(df_pca['group'], df_pca['var_3'] * 100, color='#bc8cff', edgecolor='black')
ax.axhline(80, color='#f0883e', lw=1.5, ls='--', label='80% threshold')
for bar, row in zip(bars, df_pca.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{row.n_cols} cols', ha='center', fontsize=8, color='white')
ax.set_ylabel('Variance Explained by First 3 PCs (%)')
ax.set_xlabel('V-Column NaN Group')
ax.set_title('Top 3 PCA Components Capture >80% Variance per Group → Justify PCA Reduction for DL Models')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 6 — M Column Analysis
> **Justifies:** `build_m_features()` — match counts, mismatch counts, uid aggregations

In [ ]:
m_present = [c for c in m_cols if c in train.columns]

# Encode M columns: T→1, F→0, NaN→-1
train_m = train.copy()
for m in m_present:
    train_m[m] = train_m[m].map({'T': 1, 'F': 0}).fillna(-1).astype(int)

train_m['M_match_count']    = (train_m[m_present] == 1).sum(axis=1)
train_m['M_mismatch_count'] = (train_m[m_present] == 0).sum(axis=1)
train_m['M_nan_count']      = (train_m[m_present] == -1).sum(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, color, title in zip(
    axes,
    ['M_match_count', 'M_mismatch_count', 'M_nan_count'],
    [LEGIT_COLOR, FRAUD_COLOR, '#888888'],
    ['Match Count', 'Mismatch Count', 'NaN Count']
):
    fraud_vals  = train_m.loc[train_m['isFraud'] == 1, col]
    legit_vals  = train_m.loc[train_m['isFraud'] == 0, col]
    ax.hist(legit_vals, bins=len(m_present)+1, alpha=0.6, density=True, color=LEGIT_COLOR, label='Legit')
    ax.hist(fraud_vals, bins=len(m_present)+1, alpha=0.6, density=True, color=FRAUD_COLOR, label='Fraud')
    ax.set_title(f'M Column {title} Distribution')
    ax.set_xlabel(f'Number of {title.split()[0]} Fields')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Fraud Transactions Have Fewer Matches & More Mismatches → M_match_count is a Strong Signal', color='#58a6ff')
plt.tight_layout()
plt.show()

---
## Section 7 — C Column Analysis
> **Justifies:** `build_c_features()` — ratios, C3 flag, uid aggregations

In [ ]:
c_present = [c for c in c_cols if c in train.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, col in zip(axes, c_present[:8]):
    fraud_vals = train.loc[train['isFraud'] == 1, col].dropna()
    legit_vals = train.loc[train['isFraud'] == 0, col].dropna()
    p99 = train[col].quantile(0.99)
    ax.hist(legit_vals.clip(upper=p99), bins=30, alpha=0.6, density=True,
            color=LEGIT_COLOR, label='Legit')
    ax.hist(fraud_vals.clip(upper=p99), bins=30, alpha=0.6, density=True,
            color=FRAUD_COLOR, label='Fraud')
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.legend(fontsize=7)

plt.suptitle('C-Column Distributions: Fraud vs Legit → Justifies Ratios + C3 Threshold Flag', color='#58a6ff')
plt.tight_layout()
plt.show()

# C3 flag analysis
if 'C3' in train.columns:
    c3_fraud = train.groupby('C3')['isFraud'].mean().head(15)
    fig, ax = plt.subplots(figsize=(10, 4))
    colors_c3 = [FRAUD_COLOR if i <= 3 else ACCENT for i in c3_fraud.index]
    ax.bar(c3_fraud.index.astype(str), c3_fraud.values * 100, color=colors_c3)
    ax.axvline(2.5, color='white', lw=2, ls='--', label='C3 ≤ 3 threshold')
    ax.set_title('Fraud Rate by C3 Value → C3 ≤ 3 is a Strong Fraud Signal (C3_flag feature)')
    ax.set_xlabel('C3 value')
    ax.set_ylabel('Fraud Rate (%)')
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## Section 8 — D Column Analysis
> **Justifies:** `build_d_features()` — D_norm = D - day

In [ ]:
d_present = [c for c in d_cols if c in train.columns]

# D columns are TIME-VARYING: the same account will have increasing D values over time
# D_norm = D - day stabilises this
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

if 'D1' in train.columns:
    train['D1_norm'] = train['D1'] - train['day']

    # D1 raw: increases over time (bad for model)
    axes[0,0].scatter(train['day'].sample(5000, random_state=42),
                      train['D1'].sample(5000, random_state=42),
                      alpha=0.3, s=5, color=ACCENT)
    axes[0,0].set_title('D1 RAW vs Day — Increases Over Time (Not Stable)')
    axes[0,0].set_xlabel('Day')
    axes[0,0].set_ylabel('D1 (raw)')

    # D1_norm: stable across time
    axes[0,1].scatter(train['day'].sample(5000, random_state=42),
                      train['D1_norm'].sample(5000, random_state=42),
                      alpha=0.3, s=5, color=LEGIT_COLOR)
    axes[0,1].set_title('D1_norm = D1 - day — Stable, Time-Invariant ✅')
    axes[0,1].set_xlabel('Day')
    axes[0,1].set_ylabel('D1_norm')

    # Fraud rate by D1_norm
    train['D1_norm_bin'] = pd.cut(train['D1_norm'], bins=20)
    d1_fraud = train.groupby('D1_norm_bin')['isFraud'].mean()
    axes[1,0].bar(range(len(d1_fraud)), d1_fraud.values * 100, color=ACCENT)
    axes[1,0].set_title('Fraud Rate by D1_norm Bin')
    axes[1,0].set_xlabel('D1_norm Bin')
    axes[1,0].set_ylabel('Fraud Rate (%)')

    # Distribution comparison
    axes[1,1].hist(train.loc[train['isFraud']==0,'D1_norm'].dropna().clip(-500,500),
                   bins=50, alpha=0.6, density=True, color=LEGIT_COLOR, label='Legit')
    axes[1,1].hist(train.loc[train['isFraud']==1,'D1_norm'].dropna().clip(-500,500),
                   bins=50, alpha=0.6, density=True, color=FRAUD_COLOR, label='Fraud')
    axes[1,1].set_title('D1_norm Distribution: Fraud vs Legit')
    axes[1,1].set_xlabel('D1_norm')
    axes[1,1].legend()

plt.suptitle('D Columns are Time-Varying → Normalise with D_norm = D - day', color='#58a6ff')
plt.tight_layout()
plt.show()

---
## Section 9 — Transaction Amount Analysis
> **Justifies:** `build_amount_features()` — log, decimal, z-score, ratio to uid mean

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# 1. Raw distribution: right-skewed
axes[0,0].hist(train.loc[train['isFraud']==0,'TransactionAmt'].clip(upper=2000),
               bins=60, alpha=0.6, density=True, color=LEGIT_COLOR, label='Legit')
axes[0,0].hist(train.loc[train['isFraud']==1,'TransactionAmt'].clip(upper=2000),
               bins=60, alpha=0.6, density=True, color=FRAUD_COLOR, label='Fraud')
axes[0,0].set_title('TransactionAmt RAW (right-skewed ❌)')
axes[0,0].set_xlabel('Amount ($)')
axes[0,0].legend()

# 2. Log transform: more normal
train['amt_log'] = np.log1p(train['TransactionAmt'])
axes[0,1].hist(train.loc[train['isFraud']==0,'amt_log'],
               bins=50, alpha=0.6, density=True, color=LEGIT_COLOR, label='Legit')
axes[0,1].hist(train.loc[train['isFraud']==1,'amt_log'],
               bins=50, alpha=0.6, density=True, color=FRAUD_COLOR, label='Fraud')
axes[0,1].set_title('log(1 + TransactionAmt) — More Normal ✅')
axes[0,1].set_xlabel('log(1 + Amount)')
axes[0,1].legend()

# 3. Decimal part
train['amt_decimal'] = (train['TransactionAmt'] - train['TransactionAmt'].astype(int)).round(2)
decimal_fraud = train.groupby('amt_decimal')['isFraud'].mean()
axes[0,2].bar(decimal_fraud.index, decimal_fraud.values * 100, width=0.01, color=ACCENT)
# Mark suspicious round amounts
for v in [0.0, 0.99]:
    if v in decimal_fraud.index:
        axes[0,2].axvline(v, color=FRAUD_COLOR, lw=2, ls='--')
axes[0,2].set_title('Fraud Rate by Decimal Part of Amount\n(Fraudsters use round numbers)')
axes[0,2].set_xlabel('Decimal Part')
axes[0,2].set_ylabel('Fraud Rate (%)')

# 4. Amount by ProductCD
if 'ProductCD' in train.columns:
    prod_fraud = train.groupby('ProductCD')['isFraud'].mean().sort_values(ascending=False)
    axes[1,0].bar(prod_fraud.index, prod_fraud.values * 100, color='#bc8cff')
    axes[1,0].set_title('Fraud Rate by ProductCD')
    axes[1,0].set_xlabel('Product Code')
    axes[1,0].set_ylabel('Fraud Rate (%)')

# 5. Z-score computation justification
# Sample 10 users and show their amount distribution
if 'card1' in train.columns:
    top_cards = train['card1'].value_counts().head(8).index
    for card in top_cards:
        subset = train[train['card1'] == card]['TransactionAmt']
        axes[1,1].hist(subset.clip(upper=1000), bins=20, alpha=0.3, density=True)
    axes[1,1].set_title('Amount Distribution per Card1\n(Each user has own normal → z-score needed)')
    axes[1,1].set_xlabel('Amount ($)')
    axes[1,1].set_ylabel('Density')

# 6. Global vs per-user amount statistics
if 'card1' in train.columns:
    card_stats = train.groupby('card1')['TransactionAmt'].agg(['mean', 'std', 'count'])
    card_stats = card_stats[card_stats['count'] >= 5]
    axes[1,2].scatter(card_stats['mean'], card_stats['std'],
                      alpha=0.3, s=10, color=ACCENT)
    axes[1,2].set_title('Per-Card Amount Mean vs Std\n(Different users → z-score should be per-user)')
    axes[1,2].set_xlabel('Mean Transaction Amount')
    axes[1,2].set_ylabel('Std Transaction Amount')

plt.suptitle('Amount Features: Log-Transform + Decimal + Per-User Z-Score', color='#58a6ff')
plt.tight_layout()
plt.show()

---
## Section 10 — Email & Device Analysis
> **Justifies:** `build_email_features()`, `build_device_features()`

In [ ]:
FREE_EMAIL = frozenset(['gmail.com','yahoo.com','hotmail.com','outlook.com','aol.com','icloud.com'])

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# 1. Top email domains by fraud rate
if 'P_emaildomain' in train.columns:
    email_stats = train.groupby('P_emaildomain').agg(
        fraud_rate=('isFraud','mean'), count=('isFraud','count')
    ).query('count >= 200').sort_values('fraud_rate', ascending=False).head(15)
    colors_em = [FRAUD_COLOR if d in FREE_EMAIL else ACCENT for d in email_stats.index]
    axes[0,0].barh(email_stats.index, email_stats['fraud_rate']*100, color=colors_em)
    axes[0,0].set_title('Fraud Rate by Email Domain\n(Red=Free Provider)')
    axes[0,0].set_xlabel('Fraud Rate (%)')
    axes[0,0].invert_yaxis()

# 2. Email match
if 'P_emaildomain' in train.columns and 'R_emaildomain' in train.columns:
    train['email_match'] = (train['P_emaildomain'] == train['R_emaildomain']).astype(int)
    match_fraud = train.groupby('email_match')['isFraud'].mean()
    axes[0,1].bar(['No Match (0)', 'Match (1)'], match_fraud.values * 100,
                  color=[FRAUD_COLOR, LEGIT_COLOR])
    axes[0,1].set_title('Fraud Rate by Email Match\n(P_email == R_email)')
    axes[0,1].set_ylabel('Fraud Rate (%)')
    for i, v in enumerate(match_fraud.values * 100):
        axes[0,1].text(i, v + 0.1, f'{v:.1f}%', ha='center', color='white')

# 3. Free email flag
if 'P_emaildomain' in train.columns:
    train['P_free'] = train['P_emaildomain'].isin(FREE_EMAIL).astype(int)
    free_fraud = train.groupby('P_free')['isFraud'].mean()
    axes[0,2].bar(['Non-Free', 'Free Email'], free_fraud.values * 100,
                  color=[LEGIT_COLOR, FRAUD_COLOR])
    axes[0,2].set_title('Fraud Rate: Free vs Non-Free Email Provider')
    axes[0,2].set_ylabel('Fraud Rate (%)')

# 4. Device type fraud rate
if 'DeviceType' in train.columns:
    dev_fraud = train.groupby('DeviceType')['isFraud'].mean().sort_values(ascending=False)
    axes[1,0].bar(dev_fraud.index.fillna('Unknown'), dev_fraud.values * 100, color='#4ecdc4')
    axes[1,0].set_title('Fraud Rate by Device Type')
    axes[1,0].set_ylabel('Fraud Rate (%)')

# 5. OS family extraction from DeviceInfo
if 'DeviceInfo' in train.columns:
    OS_PATTERN = re.compile(r'(Windows|iOS|Android|Mac|Linux)', re.IGNORECASE)
    train['device_os'] = train['DeviceInfo'].fillna('unknown').astype(str).str.extract(OS_PATTERN, expand=False).fillna('Other')
    os_fraud = train.groupby('device_os').agg(fraud_rate=('isFraud','mean'), count=('isFraud','count'))
    os_fraud = os_fraud[os_fraud['count'] > 100].sort_values('fraud_rate', ascending=False)
    axes[1,1].bar(os_fraud.index, os_fraud['fraud_rate']*100, color='#f0883e')
    axes[1,1].set_title('Fraud Rate by OS Family\n(Regex extracted from DeviceInfo)')
    axes[1,1].set_ylabel('Fraud Rate (%)')

# 6. Browser family
if 'id_31' in train.columns:
    BROWSER_PAT = re.compile(r'(chrome|safari|firefox|edge|ie)', re.IGNORECASE)
    train['device_browser'] = train['id_31'].fillna('unknown').astype(str).str.extract(BROWSER_PAT, expand=False).fillna('Other')
    br_fraud = train.groupby('device_browser').agg(fraud_rate=('isFraud','mean'), count=('isFraud','count'))
    br_fraud = br_fraud[br_fraud['count'] > 100].sort_values('fraud_rate', ascending=False)
    axes[1,2].bar(br_fraud.index, br_fraud['fraud_rate']*100, color='#e3b341')
    axes[1,2].set_title('Fraud Rate by Browser Family')
    axes[1,2].set_ylabel('Fraud Rate (%)')

plt.suptitle('Email & Device Features — Clear Fraud Rate Differences Justify Feature Engineering', color='#58a6ff')
plt.tight_layout()
plt.show()

---
## Section 11 — UID Construction Validation
> **Justifies:** `build_uid_features()` — Deotte trick: card1 + addr1 + (D1 - day)

In [ ]:
# ── D1 normalisation stability ────────────────────────────────────────────────
if 'D1' in train.columns and 'card1' in train.columns:
    train['D1_norm'] = (train['D1'] - train['day']).round()
    train['uid1'] = (train['card1'].astype(str) + '_' +
                     train['addr1'].astype(str) + '_' +
                     train['D1_norm'].astype(str))

    # Pick 5 cards and show their D1 vs D1_norm over time
    top5 = train['card1'].value_counts().head(5).index

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for card in top5:
        sub = train[train['card1'] == card].sort_values('day')
        axes[0].plot(sub['day'], sub['D1'], alpha=0.7, lw=1)
        axes[1].plot(sub['day'], sub['D1_norm'], alpha=0.7, lw=1)

    axes[0].set_title('D1 RAW — Grows Over Time\n(Different values same person → bad for identity)')
    axes[0].set_xlabel('Day')
    axes[0].set_ylabel('D1 (raw)')

    axes[1].set_title('D1 - day (D1_norm) — Stable Per Person ✅\n(Same value throughout = reliable identity key)')
    axes[1].set_xlabel('Day')
    axes[1].set_ylabel('D1_norm')

    plt.suptitle('The Deotte Trick: D1 - day is ~Constant Per Real User → Stable UID Key', color='#58a6ff')
    plt.tight_layout()
    plt.show()

    # UID1 statistics
    uid_stats = train.groupby('uid1').agg(count=('isFraud','count'), fraud=('isFraud','sum'))
    uid_stats['fraud_rate'] = uid_stats['fraud'] / uid_stats['count']
    print(f'Total unique uid1  : {train["uid1"].nunique():,}')
    print(f'Repeat users (≥2 txn): {(uid_stats["count"] >= 2).sum():,}')
    print(f'Users with fraud txns: {(uid_stats["fraud"] > 0).sum():,}')
    print(f'Avg transactions/user: {uid_stats["count"].mean():.1f}')

In [ ]:
# ── Why per-user aggregations matter ─────────────────────────────────────────
if 'uid1' in train.columns:
    uid_amt = train.groupby('uid1')['TransactionAmt'].agg(['mean', 'std', 'count'])
    uid_amt = uid_amt[uid_amt['count'] >= 3]

    # Merge back
    train_uid = train.merge(uid_amt.rename(columns={'mean':'uid_amt_mean','std':'uid_amt_std','count':'uid_tx_count'}),
                            on='uid1', how='left')
    train_uid['amt_zscore'] = (train_uid['TransactionAmt'] - train_uid['uid_amt_mean']) / (train_uid['uid_amt_std'].abs() + 1e-5)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(train_uid.loc[train_uid['isFraud']==0,'amt_zscore'].clip(-5,5),
                 bins=50, alpha=0.6, density=True, color=LEGIT_COLOR, label='Legit')
    axes[0].hist(train_uid.loc[train_uid['isFraud']==1,'amt_zscore'].clip(-5,5),
                 bins=50, alpha=0.6, density=True, color=FRAUD_COLOR, label='Fraud')
    axes[0].set_title('Per-User Amount Z-Score Distribution\nFraud = More Extreme Deviations ✅')
    axes[0].set_xlabel('Z-Score (std devs from user mean)')
    axes[0].legend()

    axes[1].scatter(train_uid['uid_tx_count'].clip(upper=50),
                    train_uid['uid_amt_mean'],
                    c=train_uid['isFraud'], cmap='RdYlGn_r',
                    alpha=0.3, s=10)
    axes[1].set_title('Transaction Count vs Avg Amount per User\n(Color = isFraud)')
    axes[1].set_xlabel('Transactions per User')
    axes[1].set_ylabel('Mean Amount ($)')

    plt.suptitle('Per-User Statistics Reveal Behavioral Anomalies', color='#58a6ff')
    plt.tight_layout()
    plt.show()

---
## Section 12 — Frequency Encoding Justification
> **Justifies:** `frequency_encoding.py` — why not label encoding or one-hot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Cardinality of key columns (why one-hot is impossible)
freq_cols = ['card1', 'card2', 'addr1', 'P_emaildomain', 'DeviceInfo']
freq_cols = [c for c in freq_cols if c in train.columns]
cardinalities = {c: train[c].nunique() for c in freq_cols}

axes[0].barh(list(cardinalities.keys()), list(cardinalities.values()), color=ACCENT)
axes[0].axvline(50, color=FRAUD_COLOR, lw=2, ls='--', label='One-hot limit (50)')
axes[0].set_title('Column Cardinality\n(One-hot encoding would explode dimensionality)')
axes[0].set_xlabel('Unique Values')
axes[0].legend(fontsize=8)

# 2. Frequency vs fraud rate for card1
if 'card1' in train.columns:
    card1_freq = train['card1'].value_counts(normalize=True)
    card1_fraud = train.groupby('card1')['isFraud'].mean()
    card_df = pd.DataFrame({'freq': card1_freq, 'fraud_rate': card1_fraud}).dropna()
    axes[1].scatter(card_df['freq'] * 100, card_df['fraud_rate'] * 100,
                    alpha=0.3, s=10, color=ACCENT)
    axes[1].set_title('Card1 Frequency vs Fraud Rate\n(Rare cards tend to be higher risk)')
    axes[1].set_xlabel('Card1 Frequency in Dataset (%)')
    axes[1].set_ylabel('Fraud Rate (%)')

# 3. Email domain frequency encoding — show the mapping makes sense
if 'P_emaildomain' in train.columns:
    email_freq = train['P_emaildomain'].value_counts(normalize=True).head(15)
    email_fraud = train.groupby('P_emaildomain')['isFraud'].mean()
    common_emails = email_freq.index.tolist()

    x = np.arange(len(common_emails))
    width = 0.35
    axes[2].bar(x - width/2, email_freq.values * 100, width, color=ACCENT, label='Frequency (%)', alpha=0.8)
    ax2r = axes[2].twinx()
    ax2r.bar(x + width/2, [email_fraud.get(e, 0) * 100 for e in common_emails],
             width, color=FRAUD_COLOR, label='Fraud Rate (%)', alpha=0.8)
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(common_emails, rotation=45, ha='right', fontsize=7)
    axes[2].set_title('Top Email Domains: Frequency vs Fraud Rate')
    axes[2].set_ylabel('Frequency (%)', color=ACCENT)
    ax2r.set_ylabel('Fraud Rate (%)', color=FRAUD_COLOR)

plt.suptitle('Frequency Encoding: Single Float Replaces High-Cardinality Category Without One-Hot Explosion', color='#58a6ff')
plt.tight_layout()
plt.show()

---
## Section 13 — Velocity / Behavioral Features
> **Justifies:** `velocity_features.py` — transaction bursting pattern of fraudsters

In [ ]:
# Simulate velocity features on a sample of users
import bisect

if 'uid1' in train.columns:
    # Only on users with multiple transactions
    uid_counts = train['uid1'].value_counts()
    multi_uids = uid_counts[uid_counts >= 3].index
    sample_df = train[train['uid1'].isin(multi_uids)].sort_values('TransactionDT').copy()
    sample_df = sample_df.head(50000)  # cap for speed

    # Compute gap_last
    uid_last_time = {}
    gap_last = []
    for _, row in sample_df[['uid1','TransactionDT']].iterrows():
        uid = row['uid1']
        t = row['TransactionDT']
        if uid in uid_last_time:
            gap_last.append(t - uid_last_time[uid])
        else:
            gap_last.append(-1)
        uid_last_time[uid] = t
    sample_df['gap_last'] = gap_last

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Gap distribution: fraud vs legit
    gap_legit = sample_df.loc[(sample_df['isFraud']==0) & (sample_df['gap_last']>=0), 'gap_last']
    gap_fraud = sample_df.loc[(sample_df['isFraud']==1) & (sample_df['gap_last']>=0), 'gap_last']

    axes[0].hist(np.log1p(gap_legit), bins=50, alpha=0.6, density=True, color=LEGIT_COLOR, label='Legit')
    axes[0].hist(np.log1p(gap_fraud), bins=50, alpha=0.6, density=True, color=FRAUD_COLOR, label='Fraud')
    axes[0].set_title('log(Gap Since Last Txn) by Class\nFraud = Shorter Gaps (Bursting)')
    axes[0].set_xlabel('log(seconds since last transaction)')
    axes[0].legend()

    # Transaction count per uid over time windows (proxy)
    uid_tx_per_day = sample_df.groupby(['uid1','day']).size().reset_index(name='daily_txn')
    uid_max_daily = uid_tx_per_day.groupby('uid1')['daily_txn'].max()
    uid_fraud_flag = sample_df.groupby('uid1')['isFraud'].max()
    burst_df = pd.DataFrame({'max_daily': uid_max_daily, 'is_fraud': uid_fraud_flag}).dropna()

    axes[1].hist(burst_df.loc[burst_df['is_fraud']==0,'max_daily'].clip(upper=10),
                 bins=10, alpha=0.6, density=True, color=LEGIT_COLOR, label='Legit')
    axes[1].hist(burst_df.loc[burst_df['is_fraud']==1,'max_daily'].clip(upper=10),
                 bins=10, alpha=0.6, density=True, color=FRAUD_COLOR, label='Fraud')
    axes[1].set_title('Max Transactions per Day per User\nFraud Users Have More Bursts')
    axes[1].set_xlabel('Max Daily Transaction Count')
    axes[1].legend()

    # Average gap per user vs fraud
    uid_avg_gap = sample_df[sample_df['gap_last'] >= 0].groupby('uid1')['gap_last'].mean()
    avg_gap_df = pd.DataFrame({'avg_gap': uid_avg_gap, 'is_fraud': uid_fraud_flag}).dropna()

    axes[2].boxplot(
        [np.log1p(avg_gap_df.loc[avg_gap_df['is_fraud']==0,'avg_gap'].values),
         np.log1p(avg_gap_df.loc[avg_gap_df['is_fraud']==1,'avg_gap'].values)],
        labels=['Legit', 'Fraud'],
        patch_artist=True,
        boxprops=dict(facecolor=ACCENT, color='white'),
    )
    axes[2].set_title('Average Gap Between Transactions\n(log scale) — Fraud Txns Are Bursts')
    axes[2].set_ylabel('log(avg gap in seconds)')

    plt.suptitle('Velocity Features Capture Fraudster Bursting Behavior', color='#58a6ff')
    plt.tight_layout()
    plt.show()

---
## Section 14 — Cohort Prior Analysis
> **Justifies:** `build_cohort_priors.py` — DeviceType + P_emaildomain + ProductCD cohorts

In [ ]:
COHORT_COLS = ['DeviceType', 'P_emaildomain', 'ProductCD']
cohort_cols_present = [c for c in COHORT_COLS if c in train.columns]

if len(cohort_cols_present) >= 2:
    def safe_str(val):
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return 'unknown'
        return str(val).strip().lower()

    train['cohort_key'] = train.apply(
        lambda r: '_'.join([safe_str(r.get(c, 'unknown')) for c in cohort_cols_present]), axis=1
    )

    cohort_stats = train.groupby('cohort_key').agg(
        count=('isFraud','count'),
        fraud_rate=('isFraud','mean')
    ).sort_values('count', ascending=False)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Top 15 cohorts by fraud rate
    top_fraud_cohorts = cohort_stats[cohort_stats['count'] >= 500].sort_values('fraud_rate', ascending=False).head(15)
    axes[0].barh(range(len(top_fraud_cohorts)), top_fraud_cohorts['fraud_rate']*100, color=FRAUD_COLOR)
    axes[0].set_yticks(range(len(top_fraud_cohorts)))
    axes[0].set_yticklabels(top_fraud_cohorts.index, fontsize=7)
    axes[0].invert_yaxis()
    axes[0].set_title('Top 15 High-Fraud Cohorts\n(≥500 transactions)')
    axes[0].set_xlabel('Fraud Rate (%)')

    # Cohort size distribution
    axes[1].hist(np.log10(cohort_stats['count'].clip(lower=1)), bins=30, color=ACCENT)
    axes[1].set_title(f'Cohort Size Distribution\n({len(cohort_stats)} cohorts total)')
    axes[1].set_xlabel('log10(Transaction Count)')
    axes[1].set_ylabel('Number of Cohorts')

    # Fraud rate variance across cohorts (why priors matter)
    axes[2].scatter(cohort_stats['count'], cohort_stats['fraud_rate']*100,
                    alpha=0.4, s=10, color=ACCENT)
    axes[2].axhline(train['isFraud'].mean()*100, color=FRAUD_COLOR, lw=2, ls='--',
                    label=f'Global rate: {train["isFraud"].mean():.2%}')
    axes[2].set_xscale('log')
    axes[2].set_title('Cohort Fraud Rate vs Size\nHigh Variance → Priors are Informative')
    axes[2].set_xlabel('Cohort Size (log scale)')
    axes[2].set_ylabel('Fraud Rate (%)')
    axes[2].legend()

    plt.suptitle('Cohort Priors: Different User Groups Have Very Different Fraud Rates → Cold-Start Solution', color='#58a6ff')
    plt.tight_layout()
    plt.show()

    # Check existing saved priors
    priors_path = Path('../data/processed/cohort_priors.json')
    if priors_path.exists():
        with open(priors_path) as f:
            priors = json.load(f)
        print(f'\nSaved cohort priors: {len(priors)} cohorts')
        top5 = sorted(priors.items(), key=lambda x: x[1]['count'], reverse=True)[:5]
        print('Top 5 cohorts by size:')
        for k, v in top5:
            print(f'  {k:45s}  n={v["count"]:7,}  fraud_rate={v["fraud_rate"]:.3%}')

---
## Section 15 — Feature Importance Preview (Correlation with isFraud)

In [ ]:
# Load the processed parquet to get engineered features
proc_path = Path('../data/processed/train_raw.parquet')
if proc_path.exists():
    train_proc = pd.read_parquet(proc_path)
    print(f'Processed dataset shape: {train_proc.shape}')
    print(f'Columns: {train_proc.shape[1]}')

    # Correlation of numeric cols with isFraud
    numeric_cols = train_proc.select_dtypes(include=np.number).columns.tolist()
    if 'isFraud' in numeric_cols:
        numeric_cols.remove('isFraud')

    corr_with_target = train_proc[numeric_cols].corrwith(train_proc['isFraud']).abs().dropna().sort_values(ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Top 30 features
    top30 = corr_with_target.head(30)
    colors_top = []
    for col in top30.index:
        if col.startswith('V'):
            colors_top.append('#58a6ff')
        elif col.startswith('C'):
            colors_top.append('#3fb950')
        elif col.startswith('D'):
            colors_top.append('#f0883e')
        elif col.startswith('M'):
            colors_top.append('#bc8cff')
        elif 'uid' in col or 'card' in col:
            colors_top.append('#4ecdc4')
        elif 'amt' in col or 'Amt' in col:
            colors_top.append('#e3b341')
        else:
            colors_top.append('#888888')

    axes[0].barh(top30.index, top30.values, color=colors_top)
    axes[0].invert_yaxis()
    axes[0].set_title('Top 30 Features by |Correlation| with isFraud')
    axes[0].set_xlabel('|Pearson Correlation|')

    # Distribution of all correlations
    axes[1].hist(corr_with_target.values, bins=40, color=ACCENT, edgecolor='black', lw=0.3)
    axes[1].axvline(0.1, color=FRAUD_COLOR, lw=2, ls='--', label='>0.1 strong')
    strong = (corr_with_target > 0.1).sum()
    axes[1].set_title(f'All Feature Correlations with isFraud\n{strong} features with |corr| > 0.1')
    axes[1].set_xlabel('|Pearson Correlation|')
    axes[1].set_ylabel('Feature Count')
    axes[1].legend()

    # Legend for colors
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#58a6ff', label='V-columns'),
        Patch(facecolor='#3fb950', label='C-columns'),
        Patch(facecolor='#f0883e', label='D-columns'),
        Patch(facecolor='#bc8cff', label='M-columns'),
        Patch(facecolor='#4ecdc4', label='UID/Card'),
        Patch(facecolor='#e3b341', label='Amount'),
    ]
    axes[0].legend(handles=legend_elements, loc='lower right', fontsize=8)

    plt.suptitle('Engineered Features — Which Ones Actually Correlate with Fraud?', color='#58a6ff')
    plt.tight_layout()
    plt.show()
else:
    print('Processed parquet not found — run the feature engineering pipeline first.')
    print('Expected at:', proc_path)

---
## Section 16 — Time-Based Split Validation
> **Justifies:** Why `time_split()` is better than random K-Fold

In [ ]:
split_day = int(train['day'].quantile(0.75))
train_part = train[train['day'] <= split_day]
val_part   = train[train['day'] >  split_day]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Timeline view
daily = train.groupby('day').agg(total=('isFraud','count'), fraud=('isFraud','sum'))
axes[0].fill_between(daily.index, daily['total'], alpha=0.4, color=LEGIT_COLOR, label='Train region')
axes[0].axvline(split_day, color='#f0883e', lw=3, ls='--', label=f'Split at day {split_day}')
axes[0].fill_between(daily.index[daily.index > split_day], daily.loc[daily.index > split_day,'total'],
                     alpha=0.4, color=FRAUD_COLOR, label='Val region')
axes[0].set_title('Time-Based Train/Val Split\n75% train | 25% val — NO shuffle')
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Daily Transaction Count')
axes[0].legend()

# Expanding window folds
day_min = train['day'].min()
day_max = train['day'].max()
n_folds = 5
min_train_days = 30
cut_points = np.linspace(day_min + min_train_days, day_max, n_folds + 1)

colors_folds = ['#58a6ff','#3fb950','#f0883e','#bc8cff','#4ecdc4']
y_base = 0
for i in range(n_folds):
    train_end = cut_points[i]
    val_end   = cut_points[i+1]
    axes[1].barh(i, train_end - day_min, left=day_min, color=colors_folds[i], alpha=0.7, height=0.6, label=f'Fold {i} Train')
    axes[1].barh(i, val_end - train_end, left=train_end, color=colors_folds[i], alpha=0.3, height=0.6, hatch='//')

axes[1].set_yticks(range(n_folds))
axes[1].set_yticklabels([f'Fold {i}' for i in range(n_folds)])
axes[1].set_title('Expanding Window CV\n(Solid=Train, Hatched=Val — Always forward in time)')
axes[1].set_xlabel('Day')

plt.suptitle('Time-Based Validation Prevents Data Leakage from Future → Realistic Performance Estimates', color='#58a6ff')
plt.tight_layout()
plt.show()

print(f'Train rows: {len(train_part):,}  Fraud: {train_part["isFraud"].sum():,}  Rate: {train_part["isFraud"].mean():.3%}')
print(f'Val rows  : {len(val_part):,}  Fraud: {val_part["isFraud"].sum():,}  Rate: {val_part["isFraud"].mean():.3%}')

---
## Summary — EDA → Implementation Mapping

In [ ]:
summary = {
    'EDA Finding': [
        'Identity CSV coverage is partial (only ~45% transactions have identity)',
        'Dataset is severely imbalanced (3.5% fraud)',
        'Fraud rate varies by hour / day-of-week',
        'Hour 23 and hour 0 are adjacent in reality',
        'Missingness itself is a fraud signal',
        'M-columns are T/F strings, not numeric',
        'V-columns cluster into NaN groups (same sensor block)',
        'V-columns within groups are highly correlated (>90%)',
        '3 PCA components capture >80% variance per V group',
        'C3 ≤ 3 is a strong fraud signal',
        'D-columns grow over time (time-variant)',
        'Amount is right-skewed',
        'Fraudsters use round amounts (.00 cents)',
        'Each user has own baseline amount (needs per-user z-score)',
        'Free email domains have higher fraud rates',
        'card1 has 18,000+ unique values (no one-hot)',
        'Fraudsters burst transactions in short windows',
        'Cohort fraud rate varies from 0% to 40%+',
        'Shuffled CV leaks future data into past',
    ],
    'Implementation': [
        'Left-join transaction + identity on TransactionID',
        'Use AUC-ROC as metric, not accuracy',
        'build_time_features() — is_late_night, is_weekend',
        'Cyclic sin/cos encoding for hour, day-of-week',
        'fill_nan_for_trees(-999) — trees learn missing as a state',
        'encode_m_columns(T→1, F→0, NaN→-1)',
        'group_v_cols_by_nan() — detect sensor blocks',
        'reduce_v_by_correlation(threshold=0.9)',
        'fit_v_pca(n_components=3) for DL models',
        'C3_flag = (C3 <= 3).astype(int)',
        'D_norm = D - day — time-invariant D features',
        'log1p(TransactionAmt)',
        'amt_decimal = Amt - int(Amt)',
        'amt_zscore = (Amt - uid_mean) / uid_std',
        'email_features: P_email_is_free, email_trusted',
        'frequency_encoding: card1_freq (single float)',
        'velocity_features: uid_tx_5m/1h/24h, uid_gap_last',
        'build_cohort_priors() — save per-group fraud stats',
        'time_split(q=0.75) + expanding_window_folds()',
    ]
}

df_summary = pd.DataFrame(summary)
print('EDA → Implementation Mapping')
print('=' * 100)
for _, row in df_summary.iterrows():
    print(f'  FINDING: {row["EDA Finding"]}')
    print(f'  → IMPL : {row["Implementation"]}')
    print()

In [ ]:
print('\n' + '='*60)
print('✅ EDA COMPLETE')
print('='*60)
print(f'  Dataset: {len(train):,} training transactions')
print(f'  Fraud rate: {train["isFraud"].mean():.3%}')
print(f'  V-columns: {len(v_cols)} → grouped into {len(groups)} NaN blocks')
print(f'  All preprocessing + feature engineering decisions are EDA-backed.')